In [ ]:
from google.colab import drive
import os

# 1. 드라이브 마운트 (이 과정이 없으면 파일을 못 찾습니다)
drive.mount('/content/drive')

# 2. 경로 확인
base_path = '/content/drive/MyDrive/FAERS_DATA'

if os.path.exists(base_path):
    print(f"✅ 폴더를 찾았습니다! ({base_path})")
    files = os.listdir(base_path)
    print(f"파일 개수: {len(files)}개")
    print("파일 예시:", files[:5]) # 파일 5개만 미리보기
else:
    print(f"❌ 폴더를 찾을 수 없습니다: {base_path}")
    print("내 드라이브의 폴더 목록을 확인합니다:")
    try:
        print(os.listdir('/content/drive/MyDrive'))
    except:
        print("드라이브 마운트가 제대로 되지 않았습니다.")

In [ ]:
import shutil
import os
import glob
from concurrent.futures import ThreadPoolExecutor

# 1. 원본 경로 (구글 드라이브)
source_dir = '/content/drive/MyDrive/FAERS_DATA'
# 2. 복사할 경로 (코랩 로컬 디스크 - 매우 빠름)
local_dir = '/content/faers_local'

os.makedirs(local_dir, exist_ok=True)

print(f"🚀 구글 드라이브에서 코랩 로컬({local_dir})로 파일 복사를 시작합니다...")

# 복사할 파일 패턴들
patterns = ['*.txt', '*.TXT']
files_to_copy = []
for p in patterns:
    files_to_copy.extend(glob.glob(os.path.join(source_dir, p)))

# 병렬 복사 함수 (속도 향상)
def copy_file(src):
    filename = os.path.basename(src)
    dst = os.path.join(local_dir, filename)
    if not os.path.exists(dst): # 이미 있으면 스킵
        shutil.copy(src, dst)
        return f"{filename} 복사 완료"
    return f"{filename} 이미 있음 (스킵)"

# 쓰레드 4개로 빠르게 복사
with ThreadPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(copy_file, files_to_copy))

print(f"✅ 총 {len(files_to_copy)}개 파일 복사 완료! 이제 분석이 훨씬 빨라집니다.")

# ----------------------------------------------------------
# [중요] 분석 코드의 base_path를 로컬 경로로 변경하세요!
# ----------------------------------------------------------
# base_path = '/content/drive/MyDrive/FAERS_DATA' (X) -> 이걸 지우고
base_path = '/content/faers_local'            # (O) -> 이걸로 바꾸세요

In [ ]:
import duckdb
import os
import glob
import pandas as pd

# 1. 파일 경로 설정
local_dir = '/content/faers_local'
db_path = ':memory:'

print(f"📂 '{local_dir}' 폴더 확인 중...")

# 파일 목록 확인
drug_files = sorted(glob.glob(os.path.join(local_dir, "DRUG*.txt")) +
                    glob.glob(os.path.join(local_dir, "drug*.txt")) +
                    glob.glob(os.path.join(local_dir, "DRUG*.TXT")))

if not drug_files:
    # 로컬에 없으면 드라이브 경로로 비상 전환
    print(f"⚠️ 로컬 폴더 비어있음. 구글 드라이브 원본 경로로 전환합니다.")
    local_dir = '/content/drive/MyDrive/FAERS_DATA'

print(f"✅ 데이터 경로 확정: {local_dir}")

# ---------------------------------------------------------
# 2. DuckDB 분석 시작
# ---------------------------------------------------------
print("\n🦆 데이터 분석 시작...")
con = duckdb.connect(database=':memory:')

# [수정된 부분] 줄토피(Xultophy)와 솔리쿠아(Soliqua) 추가
drug_map = {
    # 세마글루타이드 (Semaglutide)
    'ozempic': 'semaglutide', 'wegovy': 'semaglutide', 'rybelsus': 'semaglutide', 'semaglutide': 'semaglutide',

    # 터제파타이드 (Tirzepatide)
    'mounjaro': 'tirzepatide', 'zepbound': 'tirzepatide', 'tirzepatide': 'tirzepatide',

    # 리라글루타이드 (Liraglutide) - 줄토피 추가
    'saxenda': 'liraglutide', 'victoza': 'liraglutide', 'xultophy': 'liraglutide', 'liraglutide': 'liraglutide',

    # 둘라글루타이드 (Dulaglutide)
    'trulicity': 'dulaglutide', 'dulaglutide': 'dulaglutide',

    # 엑세나타이드 (Exenatide)
    'byetta': 'exenatide', 'bydureon': 'exenatide', 'exenatide': 'exenatide',

    # 릭시세나타이드 (Lixisenatide) - 솔리쿠아 추가
    'adlyxin': 'lixisenatide', 'soliqua': 'lixisenatide', 'lixisenatide': 'lixisenatide'
}

# SQL CASE문 생성
case_statements = [f"WHEN lower(drugname) LIKE '%{b}%' THEN '{i}'" for b, i in drug_map.items()]
active_ingredient_sql = f"CASE {' '.join(case_statements)} ELSE lower(drugname) END as active_ingredient"
keyword_filter = " OR ".join([f"active_ingredient = '{ing}'" for ing in set(drug_map.values())])

def load_table_safe(pattern_prefix, table_name, cols):
    # 파일 찾기
    files = sorted(glob.glob(os.path.join(local_dir, f"{pattern_prefix}*.txt")) +
                   glob.glob(os.path.join(local_dir, f"{pattern_prefix}*.TXT")) +
                   glob.glob(os.path.join(local_dir, f"{pattern_prefix.lower()}*.txt")))

    if not files:
        print(f"⚠️ {table_name} 파일이 없습니다. (Skip)")
        return False

    print(f"[{table_name}] 로드 중... ({len(files)}개 파일)")

    try:
        # 첫 번째 파일 로드 (View 생성)
        first_file = files[0]
        # 삼중 따옴표로 안전하게 쿼리 작성
        con.execute(f"""
            CREATE OR REPLACE TEMP VIEW v_tmp AS
            SELECT * FROM read_csv('{first_file}', delim='$', header=True, all_varchar=True, ignore_errors=True, quote='')
        """)

        # 컬럼 매핑 확인
        file_cols = [c[0] for c in con.execute("DESCRIBE v_tmp").fetchall()]
        select_list = []
        for req in cols:
            match = next((c for c in file_cols if c.lower() == req.lower()), None)
            select_list.append(f'"{match}" as {req}' if match else f"NULL as {req}")

        select_sql = ", ".join(select_list)
        if table_name == "drug_all": select_sql += f", {active_ingredient_sql}"

        # 테이블 생성
        con.execute(f"CREATE TABLE {table_name} AS SELECT {select_sql} FROM v_tmp")

        # 나머지 파일 병합
        for f in files[1:]:
            try:
                con.execute(f"""
                    INSERT INTO {table_name}
                    SELECT {select_sql}
                    FROM read_csv('{f}', delim='$', header=True, all_varchar=True, ignore_errors=True, quote='')
                """)
            except Exception as e:
                continue

        return True
    except Exception as e:
        print(f"❌ {table_name} 로드 중 오류: {e}")
        return False

# (1) DRUG 테이블 로드
if not load_table_safe("DRUG", "drug_all", ["primaryid", "drugname", "role_cod"]):
    raise Exception("DRUG 테이블 생성 실패. 경로를 다시 확인하세요.")

# (2) 타겟(GLP-1 PS) 추출
print("🎯 GLP-1 주 의심 약물(PS) 추출 중...")
con.execute(f"""
    CREATE TABLE target_ids AS
    SELECT DISTINCT primaryid FROM drug_all
    WHERE ({keyword_filter}) AND upper(role_cod) = 'PS'
""")
count = con.execute("SELECT count(*) FROM target_ids").fetchone()[0]
print(f"👉 분석 대상 케이스 수: {count:,}건")

if count > 0:
    # (3) 나머지 테이블 로드
    load_table_safe("DEMO", "demo_all", ["primaryid", "caseid", "age", "age_cod", "sex", "wt", "wt_cod", "rept_dt", "occp_cod", "reporter_country"])
    load_table_safe("REAC", "reac_all", ["primaryid", "pt"])
    load_table_safe("INDI", "indi_all", ["primaryid", "indi_pt"])
    load_table_safe("OUTC", "outc_all", ["primaryid", "outc_cod"])

    # (4) 최종 병합 및 저장
    print("💾 최종 병합 및 저장 중...")
    query = f"""
    WITH concomitant_agg AS (
        SELECT primaryid,
               list_aggregate(list(DISTINCT active_ingredient), 'string_agg', ';') as concomitant_drugs,
               count(DISTINCT active_ingredient) as concomitant_drug_count
        FROM drug_all
        WHERE primaryid IN (SELECT primaryid FROM target_ids) AND NOT ({keyword_filter})
        GROUP BY primaryid
    ),
    target_drug_info AS (
        SELECT primaryid, list_aggregate(list(DISTINCT active_ingredient), 'string_agg', ';') as target_drug
        FROM drug_all
        WHERE primaryid IN (SELECT primaryid FROM target_ids) AND ({keyword_filter}) AND upper(role_cod) = 'PS'
        GROUP BY primaryid
    ),
    indi_agg AS (SELECT primaryid, list_aggregate(list(indi_pt), 'string_agg', ';') as indications FROM indi_all GROUP BY primaryid),
    reac_agg AS (SELECT primaryid, list_aggregate(list(pt), 'string_agg', ';') as reaction FROM reac_all GROUP BY primaryid),
    outc_agg AS (SELECT primaryid, list_aggregate(list(outc_cod), 'string_agg', ';') as outcome FROM outc_all GROUP BY primaryid)

    SELECT d.primaryid, d.caseid, t.target_drug, d.age, d.age_cod, d.sex, d.wt, d.wt_cod, d.rept_dt, d.occp_cod as reporter_type, d.reporter_country,
           SUBSTR(CAST(d.rept_dt AS VARCHAR), 1, 4) as rept_year,
           i.indications, r.reaction, o.outcome, c.concomitant_drugs, COALESCE(c.concomitant_drug_count, 0) as concomitant_drug_count
    FROM demo_all d
    JOIN target_drug_info t ON d.primaryid = t.primaryid
    LEFT JOIN indi_agg i ON d.primaryid = i.primaryid
    LEFT JOIN reac_agg r ON d.primaryid = r.primaryid
    LEFT JOIN outc_agg o ON d.primaryid = o.primaryid
    LEFT JOIN concomitant_agg c ON d.primaryid = c.primaryid
    """

    df_result = con.execute(query).df()
    save_path = '/content/drive/MyDrive/FAERS_DATA/GLP1_final_cleaned.csv'
    df_result.to_csv(save_path, index=False)
    print(f"🎉 성공! 파일 저장 경로: {save_path}")
    print(df_result.head())
else:
    print("⚠️ 데이터는 읽었으나 GLP-1 관련 케이스가 0건입니다.")

In [ ]:
#중복제거
#최신 사례 식별자만 남김
#나라, 성별, 보고 날짜, 이상반응, 국가,

In [ ]:
# ==========================================
# 구글 코랩(Colab)용 중복 제거 코드
# ==========================================
import pandas as pd
import os
from google.colab import files # 코랩 전용 파일 처리 도구

# 1. 파일 이름 설정 (업로드한 파일명과 똑같아야 합니다!)
INPUT_FILE = 'GLP1_final_cleaned.csv'
OUTPUT_FILE = 'GLP1_deduplicated_safe.csv'

def sort_semicolon_string(s):
    """이상반응(Reaction)을 알파벳순으로 정렬하는 함수"""
    if pd.isna(s):
        return s
    parts = [p.strip() for p in str(s).split(';')]
    parts.sort()
    return ';'.join(parts)

def run_colab_deduplication():
    # 파일이 실제로 있는지 확인
    if not os.path.exists(INPUT_FILE):
        print(f"❌ 오류: '{INPUT_FILE}' 파일을 찾을 수 없습니다.")
        print("   -> 왼쪽 폴더 아이콘을 누르고 파일을 드래그해서 업로드했는지 확인해주세요.")
        return

    print(f"📥 1. 데이터 불러오는 중... ({INPUT_FILE})")
    # 대용량 파일일 수 있으므로 low_memory=False 옵션 사용
    df = pd.read_csv(INPUT_FILE, dtype=str)

    original_count = len(df)
    print(f"   -> 원본 데이터: {original_count:,} 건")

    # [전처리] 이상반응 정렬
    if 'reaction' in df.columns:
        print("⚙️ 2. 이상반응 데이터 정렬 중...")
        df['sorted_reaction'] = df['reaction'].apply(sort_semicolon_string)
    else:
        df['sorted_reaction'] = ""

    # [1단계] Case ID 기준 (최신 버전 유지)
    print("🧹 3. [1단계] Case ID 기준 정리 중...")
    # primaryid를 숫자로 변환해 정확한 최신 버전 찾기
    df['primaryid_num'] = pd.to_numeric(df['primaryid'], errors='coerce')
    df = df.sort_values(by='primaryid_num', ascending=False)

    # 같은 caseid 중 맨 위(최신)만 남김
    df_step1 = df.drop_duplicates(subset='caseid', keep='first')
    count_step1 = len(df_step1)
    print(f"   -> 1단계 완료: {count_step1:,} 건 (삭제: {original_count - count_step1:,} 건)")

    # [2단계] 핵심 정보 기준 (안전 모드)
    print("🧹 4. [2단계] 핵심 정보(성별,나이,날짜 등) 기준 정리 중...")

    # 중복 판단 기준 (날짜 포함)
    subset_cols = ['age', 'sex', 'reporter_country', 'rept_dt', 'target_drug', 'sorted_reaction']

    # 없는 컬럼은 제외하고 진행
    valid_cols = [c for c in subset_cols if c in df_step1.columns]

    df_final = df_step1.drop_duplicates(subset=valid_cols, keep='first')
    count_final = len(df_final)
    print(f"   -> 2단계 완료: {count_final:,} 건 (추가 삭제: {count_step1 - count_final:,} 건)")

    # [마무리] 저장 및 다운로드
    print(f"💾 5. 결과 저장 및 다운로드 준비... ({OUTPUT_FILE})")

    # 임시 컬럼 제거
    if 'sorted_reaction' in df_final.columns:
        del df_final['sorted_reaction']
    if 'primaryid_num' in df_final.columns:
        del df_final['primaryid_num']

    # CSV 파일로 저장
    df_final.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

    print("="*50)
    print(f"🎉 최종 완료! 총 {original_count - count_final:,} 건의 중복 제거됨.")
    print("="*50)

    # 코랩 자동 다운로드 실행
    files.download(OUTPUT_FILE)

# 실행
if __name__ == "__main__":
    run_colab_deduplication()

In [ ]:
##골든라벨 데이터셋 생성

In [ ]:
import pandas as pd
import numpy as np

# 1. 파일 로드 (이전 단계에서 만든 최종 파일)
# 경로가 다르다면 수정해주세요.
file_path = '/content/drive/MyDrive/Machine Learning/GLP1_deduplicated_safe.csv'

print("데이터 불러오는 중...")
df = pd.read_csv(file_path, dtype=str)

# ---------------------------------------------------------
# 2. 데이터 전처리 (나이, 체중, 대륙 변환)
# ---------------------------------------------------------

# (1) 숫자형 변환 및 이상치 처리
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['wt'] = pd.to_numeric(df['wt'], errors='coerce')

# (2) 나이 구간화 (Example: 0-20, 21-40, 41-60, 61+)
bins_age = [0, 20, 40, 60, 200]
labels_age = ['Age_0_20', 'Age_21_40', 'Age_41_60', 'Age_61_Plus']
df['Age_Group'] = pd.cut(df['age'], bins=bins_age, labels=labels_age, right=True)

# (3) 체중 구간화 (LBS -> KG 변환 후 구간화)
# wt_cod가 LBS인 경우 변환
mask_lbs = df['wt_cod'] == 'LBS'
df.loc[mask_lbs, 'wt'] = df.loc[mask_lbs, 'wt'] * 0.453592

bins_wt = [0, 60, 80, 100, 500]
labels_wt = ['Weight_0_60', 'Weight_61_80', 'Weight_81_100', 'Weight_101_Plus']
df['Weight_Group'] = pd.cut(df['wt'], bins=bins_wt, labels=labels_wt, right=True)

# (4) 국가 -> 대륙 매핑 (주요 국가만 매핑, 나머지는 Other)
continent_map = {
    'US': 'North America', 'CA': 'North America',
    'GB': 'Europe', 'DE': 'Europe', 'FR': 'Europe', 'IT': 'Europe', 'ES': 'Europe', 'NL': 'Europe',
    'JP': 'Asia', 'CN': 'Asia', 'KR': 'Asia', 'IN': 'Asia',
    'BR': 'South America', 'AR': 'South America',
    'AU': 'Oceania', 'NZ': 'Oceania',
    'ZA': 'Africa'
}
# 매핑되지 않은 국가는 'Other_Continent'로 처리
df['Continent'] = df['reporter_country'].map(continent_map).fillna('Not_specified_country')


# ---------------------------------------------------------
# 3. 데이터 펼치기 (Explode by PT)
# ---------------------------------------------------------
print("부작용(PT) 기준으로 데이터 펼치는 중...")
# reaction 컬럼을 리스트로 변환 후 행 단위로 분리
df['PT'] = df['reaction'].str.split(';')
df_exploded = df.explode('PT')
# 공백이나 결측 PT 제거
df_exploded = df_exploded[df_exploded['PT'].notna() & (df_exploded['PT'] != '')]


# ---------------------------------------------------------
# 4. 범주형 변수 원-핫 인코딩 (Dummies)
# ---------------------------------------------------------
print("특성별 카운트 집계 중 (오래 걸릴 수 있습니다)...")

# (1) 적응증 (Indications) - 상위 N개만 추출
# 적응증은 한 환자가 여러 개일 수 있으므로 get_dummies(sep=';') 사용
top_n = 7  # 상위 7개 적응증만 컬럼으로 만듦
all_indications = df_exploded['indications'].str.get_dummies(sep=';')
top_indications = all_indications.sum().sort_values(ascending=False).head(top_n).index
# 상위 N개에 포함되지 않는 것은 'RFU_Other'로 합치거나 제외할 수 있음 (여기선 상위 N개만 선택)
ind_dummies = all_indications[top_indications].add_prefix('RFU_')

# (2) 결과 (Outcome)
outcome_dummies = df_exploded['outcome'].str.get_dummies(sep=';')
# 이름 변경 (DE -> Outcome_Died 등)
outcome_map = {
    'DE': 'Outcome_Died', 'HO': 'Outcome_Hospitalized', 'LT': 'Outcome_Life_Threatening',
    'DS': 'Outcome_Disabled', 'CA': 'Outcome_Congenital_Anomaly', 'RI': 'Outcome_Required_Intervention',
    'OT': 'Outcome_Other'
}
outcome_dummies = outcome_dummies.rename(columns=outcome_map)

# Serious / Non-Serious 여부 추가
# Serious 조건: DE, HO, LT, DS, CA, RI 중 하나라도 있으면 1
serious_cols = ['Outcome_Died', 'Outcome_Hospitalized', 'Outcome_Life_Threatening',
                'Outcome_Disabled', 'Outcome_Congenital_Anomaly', 'Outcome_Required_Intervention']
# 존재하는 컬럼만 선택해서 확인
existing_serious_cols = [c for c in serious_cols if c in outcome_dummies.columns]
if existing_serious_cols:
    is_serious = outcome_dummies[existing_serious_cols].sum(axis=1) > 0
    outcome_dummies['Serious'] = is_serious.astype(int)
    outcome_dummies['Non_Serious'] = (~is_serious).astype(int)

# (3) 기타 단일 범주형 변수들 (Sex, Reporter, Continent, Age, Weight, Active Ingredient)
# 이들은 One-Hot Encoding
simple_dummies = pd.get_dummies(df_exploded[['sex', 'reporter_type', 'Continent', 'Age_Group', 'Weight_Group', 'target_drug']],
                                prefix=['Sex', 'Reporter', 'Continent', 'Age', 'Weight', 'Drug'])


# ---------------------------------------------------------
# 5. 최종 집계 (Group by PT)
# ---------------------------------------------------------
# 분석에 필요한 모든 더미 데이터와 PT를 합침
analysis_df = pd.concat([df_exploded[['PT']], ind_dummies, outcome_dummies, simple_dummies], axis=1)

# PT별로 그룹화하여 합계(Sum) 계산 -> 즉, 해당 PT에서 각 특성을 가진 환자 수 카운트
final_agg_df = analysis_df.groupby('PT').sum().reset_index()

print(f"집계 완료! 총 PT 개수: {len(final_agg_df):,}개")
print("결과 데이터 예시:")
display(final_agg_df.head())

# CSV 저장
save_path = '/content/drive/MyDrive/FAERS_DATA/Golden_Label_Aggregated.csv'
final_agg_df.to_csv(save_path, index=False)
print(f"파일 저장 완료: {save_path}")

In [ ]:
##21~22,23~25년도로 분리

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. 원본 파일 로드 (환자별 데이터)
file_path = '/content/drive/MyDrive/Machine Learning/GLP1_deduplicated_safe.csv'
print("데이터 로딩 중...")
df = pd.read_csv(file_path, dtype=str)

# ---------------------------------------------------------
# 2. 데이터 전처리 (공통)
# ---------------------------------------------------------
# (1) 연도(rept_year) 숫자 변환
df['rept_year'] = pd.to_numeric(df['rept_year'], errors='coerce')

# (2) 나이/체중 변환 및 구간화
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['wt'] = pd.to_numeric(df['wt'], errors='coerce')

# 이상치 처리
df.loc[(df['age'] < 0) | (df['age'] > 120), 'age'] = np.nan
mask_lbs = df['wt_cod'] == 'LBS'
df.loc[mask_lbs, 'wt'] = df.loc[mask_lbs, 'wt'] * 0.453592

# 구간화 (Binning)
df['Age_Group'] = pd.cut(df['age'], bins=[0, 20, 40, 60, 200], labels=['Age_0_20', 'Age_21_40', 'Age_41_60', 'Age_61_Plus'])
df['Weight_Group'] = pd.cut(df['wt'], bins=[0, 60, 80, 100, 500], labels=['Weight_0_60', 'Weight_61_80', 'Weight_81_100', 'Weight_101_Plus'])

# (3) 국가 -> 대륙 매핑
continent_map = {
    'US': 'North America', 'CA': 'North America',
    'GB': 'Europe', 'DE': 'Europe', 'FR': 'Europe', 'IT': 'Europe', 'ES': 'Europe', 'NL': 'Europe',
    'JP': 'Asia', 'CN': 'Asia', 'KR': 'Asia', 'IN': 'Asia',
    'BR': 'South America', 'AR': 'South America',
    'AU': 'Oceania', 'NZ': 'Oceania', 'ZA': 'Africa'
}
df['Continent'] = df['reporter_country'].map(continent_map).fillna('Other_Continent')


# ---------------------------------------------------------
# 3. 데이터 분할 (Train vs Test)
# ---------------------------------------------------------
print("연도별 데이터 분할 중...")
train_df = df[(df['rept_year'] >= 2021) & (df['rept_year'] <= 2022)].copy()
test_df = df[(df['rept_year'] >= 2023) & (df['rept_year'] <= 2025)].copy()

print(f"Train Set (21~22): {len(train_df):,}명")
print(f"Test Set (23~25): {len(test_df):,}명")


# ---------------------------------------------------------
# 4. 집계 함수 정의 (PT 기준 펼치기 + 카운트)
# ---------------------------------------------------------
def process_dataset(dataset, top_indications=None):
    # (1) PT별 행 분리 (Explode)
    dataset['PT'] = dataset['reaction'].str.split(';')
    exploded = dataset.explode('PT')
    exploded = exploded[exploded['PT'].notna() & (exploded['PT'] != '')]

    # (2) One-Hot Encoding
    # A. 적응증 (Indications)
    ind_dummies = exploded['indications'].str.get_dummies(sep=';')

    # Train셋일 경우: Top 30 선정
    if top_indications is None:
        top_indications = ind_dummies.sum().sort_values(ascending=False).head(30).index.tolist()

    # 선정된 Top 30 컬럼만 유지 (없으면 0으로 채움) -> Test셋 컬럼 통일용
    ind_features = ind_dummies.reindex(columns=top_indications, fill_value=0).add_prefix('RFU_')

    # B. 결과 (Outcome) & 심각도
    outcome_dummies = exploded['outcome'].str.get_dummies(sep=';')
    outcome_map = {'DE': 'Outcome_Died', 'HO': 'Outcome_Hospitalized', 'LT': 'Outcome_Life_Threatening',
                   'DS': 'Outcome_Disabled', 'CA': 'Outcome_Congenital_Anomaly', 'RI': 'Outcome_Required_Intervention'}
    outcome_dummies = outcome_dummies.rename(columns=outcome_map)

    # Serious 여부
    serious_cols = list(outcome_map.values())
    existing_serious = [c for c in serious_cols if c in outcome_dummies.columns]
    if existing_serious:
        is_serious = outcome_dummies[existing_serious].sum(axis=1) > 0
        outcome_dummies['Serious'] = is_serious.astype(int)
        outcome_dummies['Non_Serious'] = (~is_serious).astype(int)
    else:
        outcome_dummies['Serious'] = 0
        outcome_dummies['Non_Serious'] = 0

    # C. 기타 범주형 변수
    simple_dummies = pd.get_dummies(exploded[['sex', 'reporter_type', 'Continent', 'Age_Group', 'Weight_Group', 'target_drug']],
                                    prefix=['Sex', 'Reporter', 'Continent', 'Age', 'Weight', 'Drug'])

    # (3) PT별 합계 집계
    features = pd.concat([exploded[['PT']], ind_features, outcome_dummies, simple_dummies], axis=1)
    aggregated = features.groupby('PT').sum().reset_index()

    return aggregated, top_indications

# ---------------------------------------------------------
# 5. 실행 및 저장
# ---------------------------------------------------------
print("Train Set 집계 중...")
train_agg, train_top_ind = process_dataset(train_df, top_indications=None) # Top 30 자동 선정

print("Test Set 집계 중...")
# Train에서 뽑은 Top 30 적응증을 그대로 적용 (컬럼 순서/개수 통일)
test_agg, _ = process_dataset(test_df, top_indications=train_top_ind)

# 컬럼 순서 완벽 일치시키기 (Test셋에 Train컬럼이 없으면 0 추가, 더 있으면 삭제)
train_cols = train_agg.columns.tolist()
test_agg = test_agg.reindex(columns=train_cols, fill_value=0)

# 저장
save_dir = '/content/drive/MyDrive/Machine Learning/'
train_agg.to_csv(os.path.join(save_dir, 'Golden_Label_Train_21_22.csv'), index=False)
test_agg.to_csv(os.path.join(save_dir, 'Golden_Label_Test_23_25.csv'), index=False)

print(f"완료! 파일 저장됨:\n1. {save_dir}Golden_Label_Train_21_22.csv\n2. {save_dir}Golden_Label_Test_23_25.csv")

In [ ]:
#FAERS PT와 MedDRA 27.0 매핑

In [ ]:
import pandas as pd
import os

# 1. 파일 경로 설정 (Colab 기본 업로드 경로 기준)
llt_path = '/content/drive/MyDrive/Machine Learning/llt.asc'
pt_path = '/content/drive/MyDrive/Machine Learning/pt.asc'
train_path = '/content/drive/MyDrive/Machine Learning/Golden_Label_Train_21_22.csv'
test_path = '/content/drive/MyDrive/Machine Learning/Golden_Label_Test_23_25.csv'

# 출력 파일 경로 설정
mapped_train_path = '/content/Mapped_Golden_Label_Train.csv'
mapped_test_path = '/content/Mapped_Golden_Label_Test.csv'

# 파일 존재 여부 확인 (에러 방지용)
required_files = [llt_path, pt_path, train_path, test_path]
missing_files = [f for f in required_files if not os.path.exists(f)]
if missing_files:
    print(f"⚠️ 다음 파일이 코랩에 업로드되지 않았습니다: {missing_files}")
    print("좌측 폴더 아이콘(📁)을 눌러 파일을 먼저 업로드해 주세요.")
else:
    print("✅ 모든 파일이 확인되었습니다. 매핑을 시작합니다...\n")

    try:
        # 2. MedDRA 원본 파일 로드
        llt_cols = ['llt_code', 'llt_name', 'pt_code', 'llt_whoart_code', 'llt_harts_code',
                    'llt_costart_sym', 'llt_icd9_code', 'llt_icd9cm_code', 'llt_icd10_code',
                    'llt_currency', 'llt_jart_code', 'empty']
        pt_cols = ['pt_code', 'pt_name', 'null_field', 'pt_soc_code', 'pt_whoart_code',
                   'pt_harts_code', 'pt_costart_sym', 'pt_icd9_code', 'pt_icd9cm_code',
                   'pt_icd10_code', 'pt_jart_code', 'empty']

        # 인코딩 오류 방지를 위해 latin1 사용
        llt_df = pd.read_csv(llt_path, sep='$', encoding='latin1', header=None, names=llt_cols, on_bad_lines='skip', low_memory=False)
        pt_df = pd.read_csv(pt_path, sep='$', encoding='latin1', header=None, names=pt_cols, on_bad_lines='skip', low_memory=False)

        # 3. 필요한 컬럼 추출 및 결측치 제거
        llt_subset = llt_df[['llt_name', 'pt_code']].dropna()
        pt_subset = pt_df[['pt_code', 'pt_name']].dropna()

        # 4. PT 코드 기준으로 병합 후 딕셔너리 생성 (대소문자 무시를 위해 lower() 적용)
        merged_meddra = pd.merge(llt_subset, pt_subset, on='pt_code', how='inner')
        mapping_dict = dict(zip(merged_meddra['llt_name'].str.lower(), merged_meddra['pt_name']))

        print(f"📚 MedDRA 매핑 사전 완성! (총 {len(mapping_dict):,}개의 LLT 단어 인식 가능)")

        # 5. 사용자 데이터셋 로드
        train_df = pd.read_csv(train_path)
        test_df = pd.read_csv(test_path)

        # 매핑 함수 정의
        def map_to_pt(term):
            if not isinstance(term, str):
                return term
            term_lower = term.lower().strip()
            if term_lower in mapping_dict:
                return mapping_dict[term_lower]
            return term  # 변환 대상이 아니면 원본 유지

        # 변경 전 데이터 저장 (몇 개가 바뀌었는지 비교용)
        train_original = train_df['PT'].copy()
        test_original = test_df['PT'].copy()

        # 6. 매핑 함수 적용
        train_df['PT'] = train_df['PT'].apply(map_to_pt)
        test_df['PT'] = test_df['PT'].apply(map_to_pt)

        # 몇 개가 변경되었는지 계산
        train_changed_count = (train_original != train_df['PT']).sum()
        test_changed_count = (test_original != test_df['PT']).sum()

        # 7. 결과 저장
        train_df.to_csv(mapped_train_path, index=False)
        test_df.to_csv(mapped_test_path, index=False)

        print("\n🎉 매핑 작업이 완료되었습니다!")
        print(f"✔️ Train 데이터셋: {train_changed_count}개의 용어가 PT로 승격되었습니다.")
        print(f"✔️ Test 데이터셋: {test_changed_count}개의 용어가 PT로 승격되었습니다.")
        print(f"\n📂 결과 파일이 다음 경로에 저장되었습니다:\n- {mapped_train_path}\n- {mapped_test_path}")
        print("좌측 파일 목록에서 새로고침(↻)을 누른 후 다운로드하실 수 있습니다.")

    except Exception as e:
        print(f"❌ 오류가 발생했습니다: {e}")

In [ ]:
##매핑된 골든라벨 데이터에 SOC 컬럼 추가

In [ ]:
import pandas as pd
import os

# 1. 파일 경로 설정 (Colab 기본 업로드 경로 기준)
# 이전 단계에서 생성한 파일(Mapped_*)과 새롭게 업로드한 soc.asc, 기존의 pt.asc가 필요합니다.
pt_path = '/content/drive/MyDrive/Machine Learning/pt.asc'
soc_path = '/content/drive/MyDrive/Machine Learning/soc.asc'
train_path = '/content/drive/MyDrive/Machine Learning/Mapped_Golden_Label_Train.csv'
test_path = '/content/drive/MyDrive/Machine Learning/Mapped_Golden_Label_Test.csv'

# 출력 파일 경로
final_train_path = '/content/Final_Golden_Label_Train_with_SOC.csv'
final_test_path = '/content/Final_Golden_Label_Test_with_SOC.csv'

try:
    # 2. MedDRA PT 및 SOC 파일 로드
    pt_cols = ['pt_code', 'pt_name', 'null_field', 'pt_soc_code', 'pt_whoart_code', 'pt_harts_code',
               'pt_costart_sym', 'pt_icd9_code', 'pt_icd9cm_code', 'pt_icd10_code', 'pt_jart_code', 'empty']
    soc_cols = ['soc_code', 'soc_name', 'soc_abbrev', 'soc_whoart_code', 'soc_harts_code',
                'soc_costart_sym', 'soc_icd9_code', 'soc_icd9cm_code', 'soc_icd10_code', 'soc_jart_code', 'empty']

    # 인코딩 오류 방지를 위해 latin1 사용
    pt_df = pd.read_csv(pt_path, sep='$', encoding='latin1', header=None, names=pt_cols, on_bad_lines='skip', low_memory=False)
    soc_df = pd.read_csv(soc_path, sep='$', encoding='latin1', header=None, names=soc_cols, on_bad_lines='skip', low_memory=False)

    # 필수 컬럼만 추출
    pt_subset = pt_df[['pt_name', 'pt_soc_code']].dropna()
    soc_subset = soc_df[['soc_code', 'soc_name']].dropna()

    # 3. PT 코드의 상위 SOC 코드를 바탕으로 이름 병합
    pt_soc_merged = pd.merge(pt_subset, soc_subset, left_on='pt_soc_code', right_on='soc_code', how='inner')

    # 대소문자 무시 딕셔너리 생성 (PT_NAME -> SOC_NAME)
    mapping_dict = dict(zip(pt_soc_merged['pt_name'].str.lower(), pt_soc_merged['soc_name']))

    print(f"📚 SOC 매핑 사전 완성! (총 {len(mapping_dict):,}개의 PT-SOC 쌍 인식 가능)")

    # 4. 이전 단계에서 PT 변환이 완료된 데이터셋 로드
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    # 매핑 함수 정의
    def get_soc(term):
        if not isinstance(term, str):
            return None
        term_lower = term.lower().strip()
        # 사전에 있으면 SOC 반환, 없으면 "Unknown" 처리
        if term_lower in mapping_dict:
            return mapping_dict[term_lower]
        return "Unknown"

    # 5. SOC 매핑 적용
    train_df['SOC'] = train_df['PT'].apply(get_soc)
    test_df['SOC'] = test_df['PT'].apply(get_soc)

    # 6. 컬럼 순서 재배치: 'PT' 컬럼 바로 다음에 'SOC' 컬럼이 오도록 조정
    for df in [train_df, test_df]:
        cols = df.columns.tolist()
        cols.insert(1, cols.pop(cols.index('SOC')))
        # DataFrame 덮어쓰기 로직
        df[:] = df[cols]

    # 매핑되지 않은(Unknown) 데이터 개수 확인
    train_unmatched = (train_df['SOC'] == "Unknown").sum()
    test_unmatched = (test_df['SOC'] == "Unknown").sum()

    # 7. 최종 결과 저장
    train_df.to_csv(final_train_path, index=False)
    test_df.to_csv(final_test_path, index=False)

    print("\n🎉 SOC 매핑이 완료되었습니다!")
    print(f"✔️ Train 데이터셋: 매핑 실패 {train_unmatched}건")
    print(f"✔️ Test 데이터셋: 매핑 실패 {test_unmatched}건")
    print(f"\n📂 결과 파일이 저장되었습니다:\n- {final_train_path}\n- {final_test_path}")

except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

In [ ]:
#칸 밀림 현상 해결

In [ ]:
import pandas as pd

# 1. 밀림 현상이 발생한 파일 로드
train_df = pd.read_csv('/content/drive/MyDrive/Machine Learning/Final_Golden_Label_Train_with_SOC.csv')
test_df = pd.read_csv('/content/drive/MyDrive/Machine Learning/Final_Golden_Label_Test_with_SOC.csv')

def fix_shifted_columns(df):
    # 현재 잘못 들어간 컬럼 이름 목록 가져오기
    current_cols = list(df.columns)

    # 올바른 순서로 재배치 (맨 뒤로 밀려난 'SOC'라는 이름을 2번째 자리로 이동)
    correct_cols = current_cols.copy()
    correct_cols.remove('SOC')
    correct_cols.insert(1, 'SOC')

    # 데이터프레임의 컬럼 이름에 덮어쓰기
    df.columns = correct_cols
    return df

# 2. 복구 함수 적용
train_df = fix_shifted_columns(train_df)
test_df = fix_shifted_columns(test_df)

# 3. 올바르게 수정된 데이터 확인
print("✔️ 복구된 데이터 샘플 (Train):")
print(train_df[['PT', 'SOC', 'RFU_Product used for unknown indication']].head())

# 4. 파일 저장
train_df.to_csv('/content/Fixed_Golden_Label_Train_with_SOC.csv', index=False)
test_df.to_csv('/content/Fixed_Golden_Label_Test_with_SOC.csv', index=False)
print("\n🎉 복구 완료! Fixed_ 파일들이 저장되었습니다.")

In [ ]:
#PT와 SOC 숫자로 변환 (SOC 27개인데 28개로 나온 이유는 unknown SOC 때문)

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import json

# 1. 파일 경로 설정 (Colab 업로드 경로 기준)
# 사용자가 수동으로 수정하신 파일명을 그대로 사용합니다.
train_path = '/content/drive/MyDrive/Machine Learning/Fixed_Golden_Label_Train_with_SOC.csv'
test_path = '/content/drive/MyDrive/Machine Learning/Fixed_Golden_Label_Test_with_SOC.csv'

# 데이터 로드
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# 2. PT와 SOC 사전 만들기
# Train에만 있는 단어와 Test에만 있는 단어가 있을 수 있으므로 두 열을 합쳐서 기준을 만듭니다.
# (결측치가 있을 경우 에러가 나므로 모두 문자열(str)로 변환해줍니다.)
combined_pt = pd.concat([train_df['PT'], test_df['PT']]).astype(str)
combined_soc = pd.concat([train_df['SOC'], test_df['SOC']]).astype(str)

# 사이킷런의 LabelEncoder 초기화
le_pt = LabelEncoder()
le_soc = LabelEncoder()

# 단어들을 고유 숫자에 매핑 (학습)
le_pt.fit(combined_pt)
le_soc.fit(combined_soc)

# 3. 데이터셋에 적용 (문자 -> 숫자)
train_df['PT'] = le_pt.transform(train_df['PT'].astype(str))
test_df['PT'] = le_pt.transform(test_df['PT'].astype(str))

train_df['SOC'] = le_soc.transform(train_df['SOC'].astype(str))
test_df['SOC'] = le_soc.transform(test_df['SOC'].astype(str))

# 4. 결과 저장
# 머신러닝 모델에 바로 집어넣을 수 있는 최종 파일로 저장합니다.
final_train_path = '/content/ML_Ready_Encoded_Train.csv'
final_test_path = '/content/ML_Ready_Encoded_Test.csv'

train_df.to_csv(final_train_path, index=False)
test_df.to_csv(final_test_path, index=False)

# 5. 매핑 딕셔너리 저장
# 머신러닝이 예측한 숫자(예: 5번)가 실제로 무슨 부작용인지 사람이 알아볼 수 있도록 '정답지'를 저장해둡니다.
pt_mapping = {int(i): str(class_name) for i, class_name in enumerate(le_pt.classes_)}
soc_mapping = {int(i): str(class_name) for i, class_name in enumerate(le_soc.classes_)}

with open('/content/PT_Mapping_Dict.json', 'w', encoding='utf-8') as f:
    json.dump(pt_mapping, f, indent=4, ensure_ascii=False)
with open('/content/SOC_Mapping_Dict.json', 'w', encoding='utf-8') as f:
    json.dump(soc_mapping, f, indent=4, ensure_ascii=False)

print("🎉 성공적으로 PT와 SOC가 숫자로 인코딩되었습니다!")
print(f"✔️ Train 데이터의 기존 컬럼(예: adr_type)은 그대로 보존되었습니다.")
print(f"✔️ 변환된 고유 PT 개수: {len(le_pt.classes_):,}개")
print(f"✔️ 변환된 고유 SOC 개수: {len(le_soc.classes_):,}개")
print("\n📂 좌측 파일 탭을 새로고침(↻)하여 'ML_Ready_Encoded_Train.csv'와 'ML_Ready_Encoded_Test.csv'를 확인하세요.")

In [ ]:
#feature a, b 넣기

In [ ]:
import pandas as pd

# 1. 방금 인코딩이 완료된 파일 로드 (Colab 업로드 경로 기준)
train_path = '/content/drive/MyDrive/Machine Learning/ML_Ready_Encoded_Train.csv'
test_path = '/content/drive/MyDrive/Machine Learning/ML_Ready_Encoded_Test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# 2. 정확한 연도별 전체 환자 수 (Total N) 설정
n_train = 38649   # 2021~2022년 총 환자 수
n_test = 169919   # 2023~2025년 총 환자 수

print("🚀 Feature a, b, c, d 계산 중...")

# 3. Train 데이터셋에 계산 적용
# Feature a = 해당 부작용이 발생한 횟수 (Serious + Non_Serious)
train_df['Feature a'] = train_df['Serious'] + train_df['Non_Serious']
# Feature b = 약을 먹었지만 해당 부작용이 발생하지 않은 횟수 (Total N - Feature a)
train_df['Feature b'] = n_train - train_df['Feature a']
# Feature c, d = 외부/비교 데이터 (현재는 비교군이 없으므로 0으로 세팅)
train_df['Feature c'] = 0
train_df['Feature d'] = 0

# 4. Test 데이터셋에 계산 적용
test_df['Feature a'] = test_df['Serious'] + test_df['Non_Serious']
test_df['Feature b'] = n_test - test_df['Feature a']
test_df['Feature c'] = 0
test_df['Feature d'] = 0

# 5. 최종 결과 저장
# 이제 머신러닝에 바로 투입할 수 있는 완벽한 최종 파일입니다!
final_train_out = '/content/ML_Ready_Final_Train_ABCD.csv'
final_test_out = '/content/ML_Ready_Final_Test_ABCD.csv'

train_df.to_csv(final_train_out, index=False)
test_df.to_csv(final_test_out, index=False)

print("\n🎉 모든 변수 계산이 성공적으로 완료되었습니다!")
print(f"✔️ Train 데이터의 기존 컬럼(예: adr_type)은 그대로 보존되었습니다.")
print(f"✔️ 파일 저장 완료:\n  1) {final_train_out}\n  2) {final_test_out}")
print("\n[ Train 데이터 최종 형태 미리보기 ]")
print(train_df[['PT', 'SOC', 'adr_type', 'Feature a', 'Feature b']].head())

In [ ]:
#feature c, d 생성

In [ ]:
import pandas as pd
import os
import glob
from sklearn.preprocessing import LabelEncoder
from google.colab import drive

# 1. 구글 드라이브 연결
drive.mount('/content/drive', force_remount=True)

# ==============================================================================
# 🔢 파라미터 설정 (앞서 원본에서 추출한 정확한 환자 수 적용)
# ==============================================================================
GLP1_N_TRAIN = 38649   # 21-22년 GLP-1 총 환자 수
GLP1_N_TEST  = 169919  # 23-25년 GLP-1 총 환자 수

# 📂 파일 경로 설정
# FAERS 원본 txt 파일들이 있는 폴더 경로 (사용자 환경에 맞게 수정하세요)
faers_folder = '/content/drive/MyDrive/FAERS_DATA/'

# 우리가 직전까지 작업했던 텍스트 기반 데이터 파일 경로 (Colab 기본 경로 기준)
train_file_path = '/content/drive/MyDrive/Machine Learning/Fixed_Golden_Label_Train_with_SOC.csv'
test_file_path = '/content/drive/MyDrive/Machine Learning/Fixed_Golden_Label_Test_with_SOC.csv'
# ==============================================================================

def create_perfect_dataset():
    print("🚀 FAERS 원본 기반 c, d 계산 및 데이터셋 생성 시작...")

    # ---------------------------------------------------------
    # 1. FAERS 원본 스캔 (배경 데이터 계산)
    # ---------------------------------------------------------
    print("\n📊 1. 전체 FAERS 데이터 분석 중 (시간이 다소 소요될 수 있습니다)...")

    # (A) 전체 FAERS 환자 수 (Total N) 구하기 - DEMO 파일
    demo_files = glob.glob(os.path.join(faers_folder, "*DEMO*.txt")) + glob.glob(os.path.join(faers_folder, "*demo*.txt"))
    unique_ids = set()
    print("  👉 DEMO 파일 스캔 중 (전체 환자 수 계산)...")

    for f in demo_files:
        try:
            chunk = pd.read_csv(f, sep='$', usecols=[0], names=['primaryid'], skiprows=1, dtype=str, on_bad_lines='skip')
            unique_ids.update(chunk['primaryid'].dropna().unique())
        except: pass

    total_faers_n = len(unique_ids)
    if total_faers_n == 0:
        print("  ⚠️ 주의: DEMO 파일을 못 찾았습니다. 임시값(5천만)을 사용합니다.")
        total_faers_n = 50000000
    print(f"  ✅ 전체 FAERS 총 환자 수: {total_faers_n:,}명")

    # (B) 전체 FAERS 부작용 빈도 구하기 - REAC 파일
    reac_files = glob.glob(os.path.join(faers_folder, "*REAC*.txt")) + glob.glob(os.path.join(faers_folder, "*reac*.txt"))
    total_pt_counts = pd.Series(dtype=int)
    print("  👉 REAC 파일 스캔 중 (부작용별 빈도 계산)...")

    for f in reac_files:
        try:
            df_chunk = pd.read_csv(f, sep='$', dtype=str, on_bad_lines='skip')
            df_chunk.columns = [c.upper() for c in df_chunk.columns]
            if 'PT' in df_chunk.columns:
                counts = df_chunk['PT'].str.upper().value_counts()
                total_pt_counts = total_pt_counts.add(counts, fill_value=0)
        except: pass

    bg_dict = total_pt_counts.to_dict()
    print(f"  ✅ 부작용 빈도 집계 완료 (고유 PT {len(bg_dict):,}개 파악됨)")

    # ---------------------------------------------------------
    # 2. 내 파일에 적용 (a, b, c, d 계산)
    # ---------------------------------------------------------
    print("\n🔄 2. 내 데이터에 Feature a,b,c,d 병합 중...")
    train_df = pd.read_csv(train_file_path)
    test_df = pd.read_csv(test_file_path)

    train_df['dataset_type'] = 'train'
    test_df['dataset_type'] = 'test'
    full_df = pd.concat([train_df, test_df], ignore_index=True)

    # (A) Feature a & b
    full_df['Feature a'] = full_df['Serious'] + full_df['Non_Serious']
    full_df['GLP1_N'] = full_df['dataset_type'].map({'train': GLP1_N_TRAIN, 'test': GLP1_N_TEST})
    full_df['Feature b'] = full_df['GLP1_N'] - full_df['Feature a']

    # (C) Feature c & d
    full_df['PT_Upper'] = full_df['PT'].astype(str).str.upper()
    full_df['Count_Total'] = full_df['PT_Upper'].map(bg_dict).fillna(0)

    # c = 전체 FAERS 빈도 - 내 약물(GLP-1) 빈도
    full_df['Feature c'] = (full_df['Count_Total'] - full_df['Feature a']).clip(lower=0)
    # d = 전체 FAERS 약물 미발생 - 내 약물 미발생
    full_df['Feature d'] = (total_faers_n - full_df['GLP1_N']) - full_df['Feature c']

    # 임시 컬럼 삭제
    full_df.drop(columns=['Count_Total', 'PT_Upper', 'GLP1_N'], inplace=True)

    # ---------------------------------------------------------
    # 3. PT 및 SOC 숫자 변환 (Label Encoding)
    # ---------------------------------------------------------
    print("\n🔢 3. 머신러닝용 숫자 변환 (Label Encoding)...")
    le_pt = LabelEncoder()
    le_soc = LabelEncoder()

    full_df['PT_Code'] = le_pt.fit_transform(full_df['PT'].astype(str))
    full_df['SOC_Code'] = le_soc.fit_transform(full_df['SOC'].astype(str))

    # [중요 수정] 기존 데이터가 날아가지 않도록, 텍스트 PT, SOC만 지우고 코드로 교체
    full_df.drop(columns=['PT', 'SOC'], inplace=True)
    full_df.rename(columns={'PT_Code': 'PT', 'SOC_Code': 'SOC'}, inplace=True)

    # Train / Test 분리 및 불필요한 태그 삭제
    final_train = full_df[full_df['dataset_type'] == 'train'].drop(columns=['dataset_type'])
    final_test = full_df[full_df['dataset_type'] == 'test'].drop(columns=['dataset_type'])

    # ---------------------------------------------------------
    # 4. 최종 저장
    # ---------------------------------------------------------
    save_train_path = '/content/ML_Ready_Final_FAERS_Train.csv'
    save_test_path = '/content/ML_Ready_Final_FAERS_Test.csv'

    final_train.to_csv(save_train_path, index=False)
    final_test.to_csv(save_test_path, index=False)

    print("\n✨ 완벽하게 생성되었습니다!")
    print(f"📂 Train 저장 완료: {save_train_path}")
    print(f"📂 Test 저장 완료: {save_test_path}")
    print("\n[ Train 데이터 샘플 (adr_type 보존 확인) ]")
    print(final_train[['PT', 'SOC', 'adr_type', 'Feature a', 'Feature b', 'Feature c', 'Feature d']].head())

# 실행
create_perfect_dataset()

In [ ]:
#21-23 데이터에 adr type 추가

In [ ]:
# 1. 라이브러리 설치 (안 되어 있다면)
!pip install PyPDF2 pandas

import pandas as pd
import os
import glob
import re
import PyPDF2
from google.colab import drive

# 2. 구글 드라이브 연결
drive.mount('/content/drive', force_remount=True)

# ==============================================================================
# 📂 경로 설정 (구글 드라이브 환경에 맞춤)
# ==============================================================================
base_folder = '/content/drive/MyDrive/FDA_LABEL/'
meddra_folder = '/content/drive/MyDrive/MedDRA/'
# 💡 방금 올려주신 파일명으로 지정했습니다!
csv_filename = 'Fixed_Golden_Label_Train_with_SOC.csv'
# ==============================================================================

def load_ultimate_meddra(meddra_path):
    print("🧠 [1/3] MedDRA 의학 사전 (HLT & SMQ)을 학습 중입니다. 조금만 기다려주세요...")

    try:
        # 1. PT 로드
        pt_df = pd.read_csv(os.path.join(meddra_path, 'pt.asc'), sep='$', header=None, encoding='latin1', usecols=[0, 1])
        pt_df.columns = ['PT_CODE', 'PT_NAME']
        pt_dict = dict(zip(pt_df['PT_CODE'], pt_df['PT_NAME'].str.lower()))

        # 2. HLT 로드
        hlt_df = pd.read_csv(os.path.join(meddra_path, 'hlt.asc'), sep='$', header=None, encoding='latin1', usecols=[0, 1])
        hlt_df.columns = ['HLT_CODE', 'HLT_NAME']
        hlt_dict = dict(zip(hlt_df['HLT_CODE'], hlt_df['HLT_NAME'].str.lower()))

        # 3. HLT-PT 연결
        hlt_pt_df = pd.read_csv(os.path.join(meddra_path, 'hlt_pt.asc'), sep='$', header=None, encoding='latin1', usecols=[0, 1])
        hlt_pt_df.columns = ['HLT_CODE', 'PT_CODE']

        # 4. SMQ List 로드 (이름에서 "(SMQ)" 글자 제거하여 순수 질환명만 추출)
        smq_list_df = pd.read_csv(os.path.join(meddra_path, 'smq_list.asc'), sep='$', header=None, encoding='latin1', usecols=[0, 1])
        smq_list_df.columns = ['SMQ_CODE', 'SMQ_NAME']
        smq_list_df['SMQ_NAME'] = smq_list_df['SMQ_NAME'].str.replace(r'\s*\(SMQ\)', '', regex=True).str.lower()
        smq_dict = dict(zip(smq_list_df['SMQ_CODE'], smq_list_df['SMQ_NAME']))

        # 5. SMQ Content 로드 (어떤 PT가 어떤 SMQ에 속하는지)
        smq_content_df = pd.read_csv(os.path.join(meddra_path, 'smq_content.asc'), sep='$', header=None, encoding='latin1', usecols=[0, 1])
        smq_content_df.columns = ['SMQ_CODE', 'PT_CODE']

        # 💡 최종 매핑 딕셔너리 만들기: PT 이름 -> [관련된 HLT 및 SMQ 이름들]
        pt_to_groups = {}

        # HLT 매핑 채우기
        for _, row in hlt_pt_df.iterrows():
            pt_code, hlt_code = row['PT_CODE'], row['HLT_CODE']
            if pt_code in pt_dict and hlt_code in hlt_dict:
                pt_name = pt_dict[pt_code]
                if pt_name not in pt_to_groups: pt_to_groups[pt_name] = []
                pt_to_groups[pt_name].append(hlt_dict[hlt_code])

        # SMQ 매핑 채우기
        for _, row in smq_content_df.iterrows():
            smq_code, pt_code = row['SMQ_CODE'], row['PT_CODE']
            if pt_code in pt_dict and smq_code in smq_dict:
                pt_name = pt_dict[pt_code]
                if pt_name not in pt_to_groups: pt_to_groups[pt_name] = []
                pt_to_groups[pt_name].append(smq_dict[smq_code])

        print(f"✅ MedDRA 학습 완료! 총 {len(pt_to_groups)}개의 PT에 대해 HLT/SMQ 매핑 장착!")
        return pt_to_groups

    except Exception as e:
        print(f"❌ MedDRA 파일 읽기 오류: {e}")
        return {}


def process_fda_labels_and_map():
    print("\n🚀 [2/3] FDA 라벨 PDF 분석 시작...")

    # PDF 텍스트 추출
    pdf_files = glob.glob(os.path.join(base_folder, "*.pdf"))
    if not pdf_files:
        print("❌ 오류: FDA 라벨 PDF 파일이 폴더에 없습니다.")
        return

    full_label_text = ""
    for pdf in pdf_files:
        try:
            reader = PyPDF2.PdfReader(pdf)
            text = "".join([page.extract_text() for page in reader.pages])

            clean_text = re.sub(r'\s+', ' ', text.replace('\n', ' ').replace('\r', ' ')).upper()

            # Section 5(경고) ~ Section 7(상호작용) 추출
            start_idx = next((clean_text.find(m) for m in ["5 WARNINGS", "5. WARNINGS", "5  WARNINGS"] if clean_text.find(m) != -1), -1)
            end_idx = next((clean_text.find(m) for m in ["7 DRUG", "7. DRUG", "7  DRUG"] if clean_text.find(m) != -1), -1)

            if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
                full_label_text += clean_text[start_idx:end_idx] + " "
            else:
                full_label_text += clean_text + " "
        except Exception as e:
            print(f"⚠️ 읽기 실패: {os.path.basename(pdf)}")

    full_label_text_lower = full_label_text.lower()
    print("✅ 라벨 텍스트(부작용 섹션) 추출 완료.")

    # CSV 로드
    file_path = os.path.join(base_folder, csv_filename)
    if not os.path.exists(file_path):
        print(f"❌ 오류: CSV 파일이 없습니다 -> {csv_filename}")
        return

    print(f"\n📊 [3/3] '{csv_filename}' 데이터 채점 진행 중...")
    df = pd.read_csv(file_path)
    pt_to_groups = load_ultimate_meddra(meddra_folder)

    if 'PT' in df.columns:
        # 💡 궁극의 채점 로직
        def check_adr_ultimate(pt):
            if pd.isna(pt): return 0
            pt_str = str(pt).lower().strip()
            if pt_str == "": return 0

            # 1. 순수 PT 이름이 라벨에 있는지 확인
            if pt_str in full_label_text_lower:
                return 1

            # 2. PT가 속한 HLT나 SMQ 이름이 라벨에 있는지 확인
            if pt_str in pt_to_groups:
                for group_name in pt_to_groups[pt_str]:
                    # 예: group_name이 "acute renal failure" 일 때 라벨에 이 단어가 있다면 1점!
                    if group_name in full_label_text_lower:
                        return 1

            # 다 없으면 새로운 부작용(0)
            return 0

        df['adr_type'] = df['PT'].apply(check_adr_ultimate)

        # 통계 출력
        count_ones = df['adr_type'].sum()
        print(f"\n🎉 [결과 요약]")
        print(f"  👉 기지 부작용 (Type 1): {count_ones}개 발견 (SMQ/HLT 확장 적용)")
        print(f"  👉 신규 부작용 (Type 0): {len(df) - count_ones}개")

        # 저장
        save_path = file_path.replace('.csv', '_ULTIMATE_ADR.csv')
        df.to_csv(save_path, index=False)
        print(f"\n✨ 저장 완료! (파일명: {os.path.basename(save_path)})")
    else:
        print("❌ 오류: 파일에 'PT' 컬럼이 없습니다.")

# 실행
process_fda_labels_and_map()

In [ ]:
import pandas as pd

# 1. 파일 불러오기
file_path = '/content/drive/MyDrive/Machine Learning/Fixed_Golden_Label_Train_with_SOC_ULTIMATE_ADR.csv'
df = pd.read_csv(file_path)

# 2. 기계 결함, 의료 실수 등 부작용이 아닌 SOC(질환군) 리스트 정의
bad_socs = [
    'Injury, poisoning and procedural complications', # 과다복용, 투약오류 등
    'Product issues',                                 # 기기 고장 등
    'Surgical and medical procedures',                # 투석, 수술 등
    'Social circumstances'                            # 사회적 환경 요인
]

# 3. 해당 SOC에 속하는 PT들은 adr_type을 강제로 0으로 초기화
# (기지 부작용이 아니라, 단순 기기 문제나 실수로 처리)
df.loc[df['SOC'].isin(bad_socs), 'adr_type'] = 0

# 4. 결과 확인 및 저장
print(f"🧹 정제 완료! 남은 진짜 기지 부작용(Type 1) 개수: {df['adr_type'].sum()}개")
clean_save_path = file_path.replace('.csv', '_CLEANED.csv')
df.to_csv(clean_save_path, index=False)
print(f"✨ 최종 정제된 파일 저장 완료: {clean_save_path}")

In [ ]:
#24-25 데이터에 라벨 추가

In [ ]:
# 1. 라이브러리 설치 (이 줄이 꼭 있어야 합니다!)
!pip install PyPDF2

import pandas as pd
import os
import glob
import re
import PyPDF2
from google.colab import drive

# 2. 구글 드라이브 연결
drive.mount('/content/drive', force_remount=True)

# ==============================================================================
# 📂 경로 설정 (PDF 파일들이 있는 폴더를 정확히 지정해주세요!)
# ==============================================================================
# 예: PDF 파일들과 CSV 파일이 같은 폴더에 있다면 아래 경로를 쓰세요.
base_folder = '/content/drive/MyDrive/FDA_LABEL_25/'

# 작업할 파일 이름
csv_filename = 'Golden_Label_Test_24_25_With_SOC.csv'
# ==============================================================================

def add_adr_type_column():
    print("🚀 PDF 라벨 분석 및 adr_type 추가 시작...")

    # 3. PDF 텍스트 추출 (Section 5 & 6)
    pdf_files = glob.glob(os.path.join(base_folder, "*.pdf"))
    if not pdf_files:
        print("❌ 오류: 폴더에서 PDF 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
        return

    print(f"📄 발견된 PDF 라벨: {len(pdf_files)}개")
    full_label_text = ""

    for pdf in pdf_files:
        try:
            reader = PyPDF2.PdfReader(pdf)
            text = ""
            for page in reader.pages:
                text += page.extract_text()

            # 텍스트 정리
            clean_text = text.replace('\n', ' ').replace('\r', ' ')
            clean_text = re.sub(r'\s+', ' ', clean_text)
            upper_text = clean_text.upper()

            # Section 5(경고) ~ Section 7(약물상호작용) 사이 추출
            start_markers = ["5 WARNINGS", "5. WARNINGS", "5  WARNINGS"]
            end_markers = ["7 DRUG", "7. DRUG", "7  DRUG"]

            start_idx = -1
            for m in start_markers:
                idx = upper_text.find(m)
                if idx != -1:
                    start_idx = idx
                    break

            end_idx = -1
            for m in end_markers:
                idx = upper_text.find(m)
                if idx != -1:
                    end_idx = idx
                    break

            # 추출 (실패 시 전체 텍스트 사용)
            if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
                relevant_text = clean_text[start_idx:end_idx]
            else:
                relevant_text = clean_text

            full_label_text += relevant_text + " "

        except Exception as e:
            print(f"⚠️ 읽기 실패: {os.path.basename(pdf)}")

    # 검색용 소문자 변환
    full_label_text_lower = full_label_text.lower()
    print("✅ 라벨 텍스트 추출 완료.")


    # 4. CSV 파일 로드 및 매칭
    file_path = os.path.join(base_folder, csv_filename)
    if not os.path.exists(file_path):
        print(f"❌ 오류: 파일을 찾을 수 없습니다 ({csv_filename})")
        return

    df = pd.read_csv(file_path)
    print(f"\n📊 데이터 로딩: {len(df)}건")

    if 'PT' in df.columns:
        # adr_type 생성 (PT 이름이 라벨 텍스트에 있으면 1, 없으면 0)
        def check_adr(pt):
            if pd.isna(pt): return 0
            pt_str = str(pt).lower().strip()
            if pt_str == "": return 0
            # 부분 일치 확인 (예: 'Nausea'가 텍스트에 포함되어 있는지)
            return 1 if pt_str in full_label_text_lower else 0

        df['adr_type'] = df['PT'].apply(check_adr)

        # 결과 통계
        count_ones = df['adr_type'].sum()
        print(f"   👉 라벨에 기재된 부작용(Type 1): {count_ones}개 발견")
        print(f"   👉 미기재 부작용(Type 0): {len(df) - count_ones}개")

        # 저장
        save_path = file_path.replace('.csv', '_ADR.csv')
        df.to_csv(save_path, index=False)
        print(f"\n✨ 저장 완료! 파일명: {os.path.basename(save_path)}")
        print(f"📂 저장 위치: {save_path}")
        print("\n[결과 미리보기]")
        print(df[['PT', 'adr_type']].head())

    else:
        print("❌ 오류: 파일에 'PT' 컬럼이 없습니다.")

# 실행
add_adr_type_column()